In [15]:
import math
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from PIL import Image, UnidentifiedImageError
import pandas as pd
import numpy as np
from sklearn.utils import class_weight
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt

SPLIT_DATASET_DIR = Path("/Users/miguelcaramelo/Desktop/Data_Science/2_semestre/Deep Learning/Deep-Learning-Project/dataset/wikiart_split")



IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MIN_WIDTH = 64
MIN_HEIGHT = 64

RANDOM_STATE = 42

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

In [16]:
IMG_SIZE   = 224
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

# label_mode='categorical' -> one-hot labels (required for MixUp + Label Smoothing)
train_ds = tf.keras.utils.image_dataset_from_directory(
    SPLIT_DATASET_DIR / "train",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    label_mode="categorical"
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    SPLIT_DATASET_DIR / "val",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode="categorical"
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    SPLIT_DATASET_DIR / "test",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode="categorical"
)

class_names = train_ds.class_names
num_classes  = len(class_names)
print(f"Classes ({num_classes}): {class_names}")


Found 9336 files belonging to 23 classes.
Found 2667 files belonging to 23 classes.
Found 1335 files belonging to 23 classes.
Classes (23): ['Albrecht_Durer', 'Boris_Kustodiev', 'Camille_Pissarro', 'Childe_Hassam', 'Claude_Monet', 'Edgar_Degas', 'Eugene_Boudin', 'Gustave_Dore', 'Ilya_Repin', 'Ivan_Aivazovsky', 'Ivan_Shishkin', 'John_Singer_Sargent', 'Marc_Chagall', 'Martiros_Saryan', 'Nicholas_Roerich', 'Pablo_Picasso', 'Paul_Cezanne', 'Pierre_Auguste_Renoir', 'Pyotr_Konchalovsky', 'Raphael_Kirchner', 'Rembrandt', 'Salvador_Dali', 'Vincent_van_Gogh']


In [17]:
# Extracts labels from the dataset 
labels_list = []
for _, batch_labels in train_ds:
    # batch_labels shape: (batch_size, num_classes) — one-hot
    labels_list.extend(np.argmax(batch_labels.numpy(), axis=1))

labels_list = np.array(labels_list)

# Calculates class weights
unique_classes = np.unique(labels_list)
weights = class_weight.compute_class_weight(
    class_weight="balanced",
    classes=unique_classes,
    y=labels_list
)
class_weights = dict(enumerate(weights))
print(class_weights)

{0: np.float64(0.9997858213750268), 1: np.float64(1.3051866349783308), 2: np.float64(0.9331334332833583), 3: np.float64(1.5146009085009733), 4: np.float64(0.6206621459912246), 5: np.float64(1.3530434782608696), 6: np.float64(1.4923273657289002), 7: np.float64(1.1000353481795688), 8: np.float64(1.5317473338802297), 9: np.float64(1.4343217084037487), 10: np.float64(1.5918158567774936), 11: np.float64(1.0570652173913044), 12: np.float64(1.0824347826086957), 13: np.float64(1.4394079555966697), 14: np.float64(0.45505946578280365), 15: np.float64(1.085328993257382), 16: np.float64(1.4292712798530312), 17: np.float64(0.595180415657274), 18: np.float64(0.900028921237829), 19: np.float64(1.6043993813369994), 20: np.float64(1.0653885655597397), 21: np.float64(1.7272895467160037), 22: np.float64(0.4388249118683901)}


2026-04-07 17:32:29.066247: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [18]:
NUM_CLASSES = num_classes 

# ==========================================
# 1. PIPELINE DE DADOS (AUGMENTATION & MIXUP)
# ==========================================
# Definimos o Augmentation para correr no dataset
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
    layers.RandomBrightness(0.1),
])

def mixup(images, labels):
    """Mistura duas imagens e labels do mesmo batch. Requer labels em one-hot!"""
    batch_size = tf.shape(images)[0]
    lam = tf.random.uniform(shape=[], minval=0.0, maxval=1.0)
    lam = tf.maximum(lam, 1.0 - lam) # A primeira imagem domina

    indices = tf.random.shuffle(tf.range(batch_size))
    images2 = tf.gather(images, indices)
    labels2 = tf.gather(labels, indices)

    mixed_images = lam * images + (1.0 - lam) * images2
    mixed_labels = lam * labels + (1.0 - lam) * labels2
    return mixed_images, mixed_labels

def preprocess_train_cnn(images, labels):
    """Aplica augmentation e mixup em batches para a CNN"""
    # 1. Augmentation (A CNN espera valores [0, 1], o teu create_dataset já faz isso)
    images = data_augmentation(images, training=True)
    # 2. MixUp
    images, labels = mixup(images, labels)
    return images, labels

# ATENÇÃO: Assume-se que o teu `train_ds` e `val_ds` já estão em BATCHES 
# e que as labels já estão em ONE-HOT (como no teu create_dataset)
AUTOTUNE = tf.data.AUTOTUNE
train_ds_cnn = train_ds.map(preprocess_train_cnn, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
# Para validação não aplicamos mixup nem augmentation
val_ds_cnn = val_ds.prefetch(AUTOTUNE)


# ==========================================
# 2. MODELO CNN CUSTOMIZADO (KERAS TUNER)
# ==========================================
def build_custom_tuning_model(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(224, 224, 3)))
    
    # Repara: Já não temos camadas de RandomFlip aqui dentro! Estão no dataset.

    # --- Camadas Convolucionais Dinâmicas ---
    num_conv_layers = hp.Int('num_conv_layers', min_value=3, max_value=7)
    
    for i in range(num_conv_layers):
        model.add(layers.Conv2D(
            filters=hp.Choice(f'filters_{i}', values=[32, 64, 128]),
            kernel_size=3,
            activation='relu',
            padding='same'
        ))
        model.add(layers.BatchNormalization())
        model.add(layers.MaxPooling2D((2, 2)))
        
        if hp.Boolean(f'dropout_conv_{i}'):
            model.add(layers.Dropout(0.2))

    # --- Classificação ---
    model.add(layers.Flatten())
    model.add(layers.Dense(
        units=hp.Int('dense_units', min_value=128, max_value=512, step=128),
        activation='relu'
    ))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(NUM_CLASSES, activation='softmax'))

    # Compilação: Categorical Crossentropy suporta as labels "misturadas" do MixUp!
    lr = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy', 
        metrics=[tf.keras.metrics.F1Score(average='macro', name='f1_score')]
    )
    return model

# ==========================================
# 3. OTIMIZAÇÃO E TREINO
# ==========================================
tuner = kt.BayesianOptimization(
    build_custom_tuning_model,
    objective=kt.Objective('val_f1_score', direction='max'),
    max_trials=10,
    directory='wikiart_cnn_from_scratch',
    project_name='cnn_mixup_tuning'
)

# Callbacks para o Tuner
callbacks_list = [
    keras.callbacks.EarlyStopping(monitor='val_f1_score', mode='max', patience=5, restore_best_weights=True)
]

print("A iniciar procura automática de arquitetura com MixUp...")
# tuner.search(train_ds_cnn, validation_data=val_ds_cnn, epochs=15, callbacks=callbacks_list)

A iniciar procura automática de arquitetura com MixUp...


In [21]:


# Define o número de classes do teu subset do Wikiart
NUM_CLASSES = num_classes 

def build_tuning_model(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(224, 224, 3)))

    # --- CAMADAS CONVOLUCIONAIS (SEARCH DE 3 A 7 CAMADAS) ---
    num_conv_layers = hp.Int('num_conv_layers', min_value=3, max_value=7)
    
    for i in range(num_conv_layers):
        model.add(layers.Conv2D(
            filters=hp.Choice(f'filters_{i}', values=[32, 64, 128]),
            kernel_size=3,
            activation='relu',
            padding='same'
        ))
        model.add(layers.BatchNormalization())
        model.add(layers.MaxPooling2D((2, 2)))
        
        if hp.Boolean(f'dropout_conv_{i}'):
            model.add(layers.Dropout(0.2))

    # --- CAMADAS DE CLASSIFICAÇÃO ---
    model.add(layers.Flatten())
    
    model.add(layers.Dense(
        units=hp.Int('dense_units', min_value=128, max_value=512, step=128),
        activation='relu'
    ))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(NUM_CLASSES, activation='softmax'))

    # --- COMPILAÇÃO COM F1-SCORE ---
    # Usamos o F1-Score Macro para dar igual peso a todas as classes, 
    # ignorando o facto de haver autores com muito mais quadros que outros.
    lr = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=[
            # Definimos o nome 'f1_score' para que o Tuner o consiga encontrar
            tf.keras.metrics.F1Score(average='macro', name='f1_score')
        ]
    )
    return model

# --- CONFIGURAÇÃO DA OTIMIZAÇÃO BAYESIANA ---
tuner = kt.BayesianOptimization(
    build_tuning_model,
    # O objetivo agora é maximizar o F1-Score nos dados de validação
    objective=kt.Objective('val_f1_score', direction='max'), 
    max_trials=10, # Testa 10 arquiteturas diferentes
    directory='diretorio_tuner_wikiart',
    project_name='cnn_f1_optimization'
)

# Resumo do espaço de hiperparâmetros que vai ser testado
tuner.search_space_summary()

# --- EXECUÇÃO DA PROCURA ---
# Descomenta a linha abaixo para iniciares o treino, passando os teus datasets de treino e validação.
tuner.search(train_ds, validation_data=val_ds, epochs=10)

# --- EXTRAÇÃO DOS RESULTADOS ---
best_model = tuner.get_best_models(num_models=1)[0]
best_hyperparameters = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Melhor número de camadas convolucionais: {best_hyperparameters.get('num_conv_layers')}")

Trial 10 Complete [00h 22m 08s]
val_f1_score: 0.3149231970310211

Best val_f1_score So Far: 0.3325437009334564
Total elapsed time: 05h 03m 57s
Melhor número de camadas convolucionais: 6


/Users/miguelcaramelo/anaconda3/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 58 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
